# 13. Context interventions and identity geometry

![Context interventions: hold identity fixed, change context, then test the geometry](../images/13_context_interventions.svg)

Lesson 12 measured how close two things are. This notebook asks the harder question: when that number changes, what caused the change?

**Learning goals:** create a paired context substitution, measure normalized loss change and pairwise geometry distortion, evaluate identity with enrollment-only centroids and cosine retrieval, and compare hard with soft completion.

In [ ]:
from itertools import product
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp

SEED = 13
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

def normalize_rows(z, eps=1e-12):
    z = np.asarray(z, dtype=float)
    norms = np.linalg.norm(z, axis=1, keepdims=True)
    if np.any(norms <= eps):
        raise ValueError("cosine geometry requires nonzero row vectors")
    return z / norms

print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Generate paired identity and context observations

A context intervention needs matched rows, so the simulation builds them explicitly rather than sampling two independent sets.

Each of five identities gets a latent direction in four dimensions. For every identity and repeat, the baseline and intervention rows share the same identity center and the same noise draw, and the intervention adds a fixed context shift on top. Sharing the noise is what isolates the intended context change here; in real data that role is played by the substitution design.

Arrays `Z_base` and `Z_intervention` both have shape `(n_pairs, representation_dim)`, and row $i$ of one corresponds to row $i$ of the other. Shuffling either array independently would destroy the experiment while leaving every shape intact.

In [ ]:
n_id, repeats, p = 5, 8, 4
identity_centers = normalize_rows(rng.normal(size=(n_id, p)))
labels = np.repeat(np.arange(n_id), repeats)
paired_noise = rng.normal(0, 0.08, size=(n_id * repeats, p))
context_shift = np.array([0.65, -0.30, 0.15, 0.20])
Z_base = identity_centers[labels] + paired_noise
Z_intervention = identity_centers[labels] + paired_noise + context_shift
assert Z_base.shape == Z_intervention.shape == (40, 4)
assert np.array_equal(labels[:repeats], np.zeros(repeats, dtype=int))
print("paired representations:", Z_base.shape, "context shift norm:", round(np.linalg.norm(context_shift), 3))

## 2. Loss contrasts and geometry matching

Two measurements answer two different questions about the same pair of arrays.

Loss asks how performance moved against a target. Here loss is squared distance to the correct identity center, and the symmetric contrast $(L_i-L_b)/((|L_i|+|L_b|)/2+\epsilon)$ divides by the local scale so that a change of 0.1 is read differently when baseline loss is 0.05 than when it is 10. Its sign flips if you swap the conditions, and nothing else changes.

Geometry matching instead asks whether the examples kept their relative positions. It compares upper-triangular pairwise cosine distances before and after, so a shared rotation of every embedding leaves it at zero even though every coordinate moved. Cosine geometry rejects zero-norm rows explicitly, because a collapsed vector has no direction to compare. Pairwise matrices cost $O(n^2)$ memory, so sample a fixed, seeded set of pairs for large datasets.

In [ ]:
def normalized_contrast(base, intervention, eps=1e-12):
    scale = (np.abs(base) + np.abs(intervention)) / 2
    return (intervention - base) / (scale + eps)

def cosine_matrix(z):
    z_normalized = normalize_rows(z)
    return 1 - np.clip(z_normalized @ z_normalized.T, -1, 1)

loss_base = np.sum((Z_base - identity_centers[labels]) ** 2, axis=1)
loss_intervention = np.sum((Z_intervention - identity_centers[labels]) ** 2, axis=1)
contrast = normalized_contrast(loss_base, loss_intervention)
upper = np.triu_indices(len(labels), k=1)
Db, Di = cosine_matrix(Z_base), cosine_matrix(Z_intervention)
geometry_mae = np.mean(np.abs(Db[upper] - Di[upper]))
assert np.allclose(np.diag(Db), 0, atol=1e-12)
collapsed_rejected = False
try:
    cosine_matrix(np.zeros((len(labels), p)))
except ValueError:
    collapsed_rejected = True
assert collapsed_rejected
print(f"mean normalized loss contrast={contrast.mean():.3f}")
print(f"pairwise geometry MAE={geometry_mae:.3f}; zero collapse rejected={collapsed_rejected}")

## 3. Separate enrollment from probes

Geometry stability alone proves nothing: a collapsed representation is perfectly stable. The informativeness test is whether identity can still be read out, and that test is only honest when the probe never helped build its own reference.

The first four repeats per identity form enrollment in the baseline context. The remaining four paired rows form probes in the intervention context, so no probe contributes to any centroid.

For cosine centroids the order of operations matters. Normalize each enrollment row before averaging, so high-norm rows do not receive extra weight, then normalize each centroid again. In real data, split participant, sequence, or source groups rather than nearby rows, because adjacent windows of one recording leak just as effectively as duplicated rows.

In [ ]:
repeat_index = np.tile(np.arange(repeats), n_id)
enroll_mask = repeat_index < repeats // 2
probe_mask = ~enroll_mask
assert not np.any(enroll_mask & probe_mask)
assert enroll_mask.sum() == probe_mask.sum() == 20

def fit_cosine_centroids(z, y):
    classes = np.unique(y)
    unit_z = normalize_rows(z)
    centroids = np.stack([unit_z[y == k].mean(axis=0) for k in classes])
    return classes, normalize_rows(centroids)

classes, centroids = fit_cosine_centroids(Z_base[enroll_mask], labels[enroll_mask])
probe_z, probe_y = Z_intervention[probe_mask], labels[probe_mask]
pred = classes[np.argmax(normalize_rows(probe_z) @ centroids.T, axis=1)]
centroid_accuracy = np.mean(pred == probe_y)
assert np.allclose(np.linalg.norm(centroids, axis=1), 1.0)
print(f"cross-context nearest-centroid accuracy={centroid_accuracy:.3f}")

## 4. Individual cosine retrieval

A centroid compresses each identity into one vector, which is efficient and lossy. Retrieval keeps every enrollment item, so it can still match an identity that occupies two separate regions.

After one row normalization, a single matrix multiplication returns all probe-to-gallery similarities. `np.ascontiguousarray` stores the transposed gallery in a BLAS-friendly memory layout, which pays off when the same gallery is reused across many probe batches.

A probe succeeds when its nearest enrollment item carries the same identity. This synthetic example has continuous noise, so exact ties are unlikely; Lesson 12 supplies the tie-aware metrics for when they do occur.

In [ ]:
gallery_z, gallery_y = Z_base[enroll_mask], labels[enroll_mask]
gallery_transpose = np.ascontiguousarray(normalize_rows(gallery_z).T)
similarity = normalize_rows(probe_z) @ gallery_transpose
nearest = np.argmax(similarity, axis=1)
retrieval_accuracy = np.mean(gallery_y[nearest] == probe_y)
assert similarity.shape == (len(probe_z), len(gallery_z))
print(f"cross-context retrieval accuracy={retrieval_accuracy:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(similarity, aspect="auto", cmap="viridis")
ax.set(xlabel="enrollment item", ylabel="intervention probe", title="Cosine similarity")
fig.colorbar(image, ax=ax, label="similarity")
plt.tight_layout()
plt.show()

## 5. Generic hard and soft expected-loss completion

The evaluations above assumed complete inputs. When a partial query leaves one binary context unknown, the missing value has to be filled in before scoring, and the filling rule changes the answer.

Hard completion selects the most probable context and discards the rest. Soft completion keeps both possibilities and averages their losses. Expected loss is generally the safer summary, because a nonlinear loss does not commute with expectation: the loss at the mean representation is not the mean of the losses, which is exactly what the final assertion checks.

This is the generic decision rule. It is not the study's independent-factor completion control, which the next section defines.

In [ ]:
probabilities = np.array([0.60, 0.40])
candidate_representations = np.array([[1.0, 0.0], [-1.0, 0.0]])
target = np.array([1.0, 0.0])
candidate_losses = np.sum((candidate_representations - target) ** 2, axis=1)
hard_index = np.argmax(probabilities)
hard_loss = candidate_losses[hard_index]
expected_loss = probabilities @ candidate_losses
mean_representation = probabilities @ candidate_representations
loss_at_mean = np.sum((mean_representation - target) ** 2)
assert not np.isclose(expected_loss, loss_at_mean)
print(f"hard loss={hard_loss:.2f}; expected loss={expected_loss:.2f}; loss at mean={loss_at_mean:.2f}")

## 6. GFC-v2 independent-factor completion

The study control answers a skeptic: could a model score well on GFC-v2 by predicting each factor separately, without recombining donor-supplied blocks at all?

It reuses the raw speed, clothing, and direction score pairs from the same donor-supplied query blocks, and the target again contributes no score block. Each complete participant supplies exactly 16 allowed target and focal-factor queries, so the control and GFC-v2 are measured on identical queries.

Hard completion takes each marginal winner and splits mass across tied marginal maxima. Soft completion applies one positive development-fitted temperature and adds marginal log probabilities for each complete gallery cell, which is the same as multiplying three probabilities but stays finite for extreme scores. In a complete Cartesian gallery the hard and soft maximizing sets coincide under the same tolerance, and the assertions below check that invariant at several temperatures. Soft target NLL and calibration still move with temperature, which is why they are reported separately.

For training condition $h$ and replicate regime $r$, the completion gap is $G_{h,r}=Y_{h,r}-C_{h,r}$. The prespecified contrast is $J=(G_{H,R}-G_{L,R})-(G_{H,F}-G_{L,F})$. With $\delta_G=0.0625$, $J$ is resolved when its 95% confidence interval excludes zero and its magnitude reaches the margin. It is practically equivalent when its 90% interval lies inside $[-\delta_G,\delta_G]$. Otherwise it is inconclusive. Even a resolved result does not prove intrinsic composition or unsupervised factor discovery, because both endpoints use supervised alignment, share errors, and can be compressed by the bounded top-1 scale.

In [ ]:
factor_names = ("speed", "clothing", "direction")
complete_gallery = list(product([0, 1], repeat=3))
recordings = {}
for cell in complete_gallery:
    raw_scores = []
    for level in cell:
        pair = np.array([-0.4, -0.4], dtype=np.float64)
        pair[level] = 1.6
        raw_scores.append(pair)
    recordings[cell] = {
        "source_video_id": f"source-speed{cell[0]}-clothing{cell[1]}",
        "raw_scores": np.stack(raw_scores),
    }

def completion_donors(target, focal_index):
    donor_u = tuple(value if j == focal_index else 1 - value
                    for j, value in enumerate(target))
    donor_v = tuple(1 - value if j == focal_index else value
                    for j, value in enumerate(target))
    return donor_u, donor_v

def require_source_separation(target, donor_u, donor_v):
    target_source = recordings[target]["source_video_id"]
    if any(recordings[donor]["source_video_id"] == target_source
           for donor in (donor_u, donor_v)):
        raise ValueError("donor and target source_video_id must differ")

def build_completion_query(target, focal_index):
    donor_u, donor_v = completion_donors(target, focal_index)
    require_source_separation(target, donor_u, donor_v)
    score_sources = [donor_u if j == focal_index else donor_v for j in range(3)]
    assert target not in score_sources
    return {
        "target": target,
        "focal_index": focal_index,
        "donors": (donor_u, donor_v),
        "score_sources": tuple(score_sources),
        "raw_scores": np.stack([recordings[source]["raw_scores"][j]
                                for j, source in enumerate(score_sources)]),
    }

queries = [build_completion_query(target, focal_index)
           for target in complete_gallery for focal_index in (0, 1)]
assert len(queries) == 16
assert all(donor in complete_gallery for query in queries for donor in query["donors"])
assert all(source != query["target"]
           for query in queries for source in query["score_sources"])
assert all(sum(query["target"] == target for query in queries) == 2
           for target in complete_gallery)

direction_rejections = 0
for target in complete_gallery:
    try:
        require_source_separation(target, *completion_donors(target, 2))
    except ValueError:
        direction_rejections += 1
assert direction_rejections == 8

def target_credit(log_mass, target_index, tolerance=1e-12):
    top = np.abs(log_mass - np.max(log_mass)) <= tolerance
    return float(top[target_index]) / int(top.sum()), top

def hard_completion(query, tolerance=1e-12):
    log_marginal = np.full((3, 2), -np.inf, dtype=np.float64)
    for j, pair in enumerate(query["raw_scores"]):
        tied = np.abs(pair - np.max(pair)) <= tolerance
        log_marginal[j, tied] = -np.log(int(tied.sum()))
    gallery_log_mass = np.array([log_marginal[np.arange(3), cell].sum()
                                 for cell in complete_gallery], dtype=np.float64)
    return target_credit(gallery_log_mass, complete_gallery.index(query["target"]),
                         tolerance)

def soft_completion(query, temperature, tolerance=1e-12):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    scaled = query["raw_scores"] / temperature
    log_marginal = scaled - logsumexp(scaled, axis=1, keepdims=True)
    gallery_log_mass = np.array([log_marginal[np.arange(3), cell].sum()
                                 for cell in complete_gallery], dtype=np.float64)
    target_index = complete_gallery.index(query["target"])
    credit, top = target_credit(gallery_log_mass, target_index,
                                tolerance / temperature)
    return credit, -gallery_log_mass[target_index], np.exp(gallery_log_mass.max()), top

hard = [hard_completion(query) for query in queries]
hard_top1 = np.mean([result[0] for result in hard])

def aggregate_soft(temperature):
    results = [soft_completion(query, temperature) for query in queries]
    top1 = np.mean([result[0] for result in results])
    nll = np.mean([result[1] for result in results])
    confidence = np.mean([result[2] for result in results])
    return top1, nll, abs(top1 - confidence), results

cold = aggregate_soft(0.7)
warm = aggregate_soft(2.0)
assert hard_top1 == cold[0] == warm[0] == 1.0
assert all(np.array_equal(hard_result[1], cold_result[3])
           for hard_result, cold_result in zip(hard, cold[3]))
assert not np.isclose(cold[1], warm[1])
assert not np.isclose(cold[2], warm[2])

tied_query = dict(queries[0])
tied_query["raw_scores"] = queries[0]["raw_scores"].copy()
tied_query["raw_scores"][0] = np.array([1.0, 1.0])
hard_tie_credit, hard_tie_set = hard_completion(tied_query)
soft_tie_credit, _, _, soft_tie_set = soft_completion(tied_query, 0.7)
assert hard_tie_credit == soft_tie_credit == 0.5
assert np.array_equal(hard_tie_set, soft_tie_set)
near_tie_query = dict(queries[0])
near_tie_query["raw_scores"] = queries[0]["raw_scores"].copy()
near_tie_query["raw_scores"][0] = np.array([1.0, 1.0 + 0.5e-12])
hard_near_credit, hard_near_set = hard_completion(near_tie_query)
for temperature in (0.5, 1.0, 2.0):
    soft_near_credit, _, _, soft_near_set = soft_completion(near_tie_query, temperature)
    assert soft_near_credit == hard_near_credit == 0.5
    assert np.array_equal(soft_near_set, hard_near_set)

extreme_query = dict(queries[0])
extreme_query["raw_scores"] = np.array([[-1000.0, 1000.0]] * 3)
extreme_nll = soft_completion(extreme_query, 1.0)[1]
assert np.isfinite(extreme_nll) and np.isclose(extreme_nll, 6000.0)

def completion_gap_interaction(gaps):
    return (gaps["H", "R"] - gaps["L", "R"]
            - gaps["H", "F"] + gaps["L", "F"])

def interpret_j(point, ci95, ci90, delta_g=0.0625):
    excludes_zero = ci95[1] < 0 or ci95[0] > 0
    if excludes_zero and abs(point) >= delta_g:
        return "resolved"
    if ci90[0] > -delta_g and ci90[1] < delta_g:
        return "practically equivalent"
    return "inconclusive"

delta_g = 0.0625
example_gaps = {("L", "F"): 0.01, ("H", "F"): 0.03,
                ("L", "R"): 0.00, ("H", "R"): 0.10}
assert np.isclose(completion_gap_interaction(example_gaps), 0.08)
assert interpret_j(0.08, (0.02, 0.14), (0.03, 0.13)) == "resolved"
assert interpret_j(0.01, (-0.04, 0.05), (-0.03, 0.04)) == "practically equivalent"
assert interpret_j(0.04, (-0.01, 0.09), (-0.01, 0.08)) == "inconclusive"
assert interpret_j(0.00, (-0.08, 0.08), (-delta_g, 0.03)) == "inconclusive"
assert interpret_j(0.00, (-0.08, 0.08), (-0.03, delta_g)) == "inconclusive"

print(f"16-query hard and soft top-1={hard_top1:.1f}; tied credit={hard_tie_credit:.1f}")
print(f"T=0.7: NLL={cold[1]:.3f}, one-bin calibration error={cold[2]:.3f}")
print(f"T=2.0: NLL={warm[1]:.3f}, one-bin calibration error={warm[2]:.3f}")
print("direction-focal source violations rejected:", direction_rejections)

## Exercises, construct validity, and takeaways

1. Set `context_shift` to zero. Predict the loss contrast and geometry error.
2. Fit centroids using all rows, including probes. Why is the resulting accuracy invalid?
3. Increase the rare completion's loss and compare hard with expected loss.

**Brief answers:** a zero shift gives contrast and geometry error near zero. Probe-fitted centroids leak evaluation information. Hard completion remains fixed while expected loss grows with the rare but costly outcome.

**Construct validity:** invariance is meaningful only if identity remains decodable and the intervention truly changes context. A zero-vector collapse is outside the cosine domain and must be rejected, not scored as perfect stability. Add a positive context-sensitive control, a negative control, and group-separated enrollment and probes.

**Takeaway:** loss, pairwise geometry, centroids, and retrieval expose different kinds of context sensitivity, and they are most informative when they disagree. The GFC-v2 hard and soft controls share top-1 in a complete gallery, while soft NLL and calibration retain confidence information. None of these measurements replaces a valid intervention design.

## Continue learning

[Previous notebook: 12](12_blockwise_distances_and_ranking.ipynb) | [Lecture](../lectures/13_context_interventions.md) | [Curriculum](../README.md) | [Next notebook: 14](14_paired_inference.ipynb)